In [ ]:
import networkx as nx
import pygraphviz as gv

_ = h.load_file("RGCmodelGD.hoc")
RGC = h.DSGC(0, 0)
soma = RGC.soma
all_dends = RGC.dend

def dist_graph(rgc):
    """Node indices correspond to (dend index + 1) as the soma is the 0th node. e.g. RGC.dend[0] is node 1
    on the graph. Distances are between the centres (0.5) of each section corresponding to synapse locations."""
    dg = nx.DiGraph()
    
    secs = [rgc.soma] + [d for d in rgc.dend]  # soma takes 0th position, offseting all dends by 1
    edges = []
    for i, parent in enumerate(secs):
        parent_ref = h.SectionRef(parent)
        for j in range(parent_ref.nchild()):
            child = parent_ref.child[j]
            # dend names are formated as "DSGC[1].dend[17]" and the id corresponds to the index
            # in the rgc.dend list of all dendrites
            cid = int(child.name().split(".")[1][5:-1]) + 1  # offset to accomodate 0th soma node
            edges.append((i, cid, h.distance(parent(0.5), child(0.5))))
    dg.add_weighted_edges_from(edges)
    return dg
    
dg = dist_graph(RGC)

In [90]:
ego = nx.generators.ego_graph(dg, 0, radius=50, distance="weight")

In [83]:
adg = nx.nx_agraph.to_agraph(dg)

In [86]:
adg.draw("/home/geoff/test.svg", prog="neato")

In [91]:
nx.nx_agraph.to_agraph(ego).draw("/home/geoff/test_ego.svg", prog="neato")